# Dog Bounding Box Detection
Uses a pre-trained YOLOv5 model to detect dogs and draw bounding boxes.

In [6]:
import os
import json
import time
import re
import math
from collections import Counter

import cv2
import numpy as np
from ultralytics import YOLO
import pandas as pd

### Methods

In [7]:
video_folder = 'files/data'
output_folder = 'files/yolo26-outputs'

# Load YOLOv26 model
model = YOLO("yolo26n.pt")
model_name = 'yolo26'

# Load font
#font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 40)

In [8]:
def get_anchor_times(duration, target_coverage=0.40, window_size=2.0):
    if duration <= 30:
        # short videos: fixed important points
        anchor_times = [
            0.10 * duration,
            0.25 * duration,
            0.50 * duration,
            0.75 * duration,
            0.90 * duration
        ]
        video_type = "short"

    elif duration <= 120:
        # medium videos: 40% coverage spread across video
        num_anchors = int(np.ceil((target_coverage * duration) / window_size))

        anchor_times = np.linspace(
            window_size / 2,
            duration - window_size / 2,
            num_anchors
        ).tolist()

        # force early/middle/late checks
        required = [
            5,
            15,
            30,
            0.10 * duration,
            0.50 * duration,
            0.90 * duration
        ]

        required = [t for t in required if 0 < t < duration]

        anchor_times.extend(required)
        anchor_times = sorted(set(round(t, 2) for t in anchor_times))
        video_type = "medium"

    else:
        # long videos, including your ~600 second videos
        long_coverage = 0.40

        num_anchors = int(np.ceil((long_coverage * duration) / window_size))

        anchor_times = np.linspace(
            window_size / 2,
            duration - window_size / 2,
            num_anchors
        ).tolist()

        # force early checks because 10% of 600s = 60s, which is too late
        required = [
            5,
            15,
            30,
            0.10 * duration,
            0.25 * duration,
            0.50 * duration,
            0.75 * duration,
            0.90 * duration
        ]

        required = [t for t in required if 0 < t < duration]

        anchor_times.extend(required)
        anchor_times = sorted(set(round(t, 2) for t in anchor_times))
        video_type = "long"

    return video_type, anchor_times

I added this 

In [9]:
def estimate_num_dogs_yolo26(
    video_path,
    model,
    duration=None,
    coverage=0.40,
    window_size=2.0,
    frames_per_anchor=10,
    conf_threshold=0.25,
    max_anchors=None,
    device=0
):
    """
    Estimates number of visible dogs in a video using YOLOv26 sampled windows.

    Returns:
        final_count: max stable dog count across sampled windows
        debug_info: useful details for checking runtime and behavior
    """

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return -1, {
            "error": "could_not_open_video",
            "video_path": video_path
        }

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if duration is None or duration <= 0:
        duration = total_frames / fps if fps > 0 else 0

    if duration <= 0 or fps <= 0:
        cap.release()
        return -1, {
            "error": "bad_duration_or_fps",
            "video_path": video_path,
            "fps": fps,
            "total_frames": total_frames
        }

    video_type, anchors = get_anchor_times(
        duration=duration,
        target_coverage=coverage,
        window_size=window_size
    )

    window_stable_counts = []
    total_yolo_frames = 0

    def stable_count(counts, min_fraction=0.30, min_frames=3):
        if len(counts) == 0:
            return 0

        threshold = max(min_frames, math.ceil(min_fraction * len(counts)))
        freq = Counter(counts)

        for count in sorted(freq.keys(), reverse=True):
            if freq[count] >= threshold:
                return count

        return 0
    
    for anchor in anchors:
        start_t = max(0, anchor - window_size / 2)
        end_t = min(duration, anchor + window_size / 2)

        # Sample frames evenly inside this 2-second window
        sample_times = np.linspace(start_t, end_t, frames_per_anchor)

        frame_counts = []

        for t in sample_times:
            frame_idx = int(t * fps)
            frame_idx = min(max(frame_idx, 0), total_frames - 1)

            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()

            if not ret:
                continue

            results = model(frame, verbose=False, device=device, imgsz=1280, conf=0.05)
            r = results[0]

            dog_count = 0

            if r.boxes is not None:
                cls_ids = r.boxes.cls.cpu().numpy()
                confs = r.boxes.conf.cpu().numpy()

                for cls_id, conf in zip(cls_ids, confs):
                    label = model.names[int(cls_id)]
                    if label == "dog" and conf >= conf_threshold:
                        dog_count += 1

            frame_counts.append(dog_count)
            total_yolo_frames += 1

        window_count = stable_count(frame_counts)
        window_stable_counts.append(window_count)

    cap.release()

    final_count = max(window_stable_counts) if window_stable_counts else 0

    debug_info = {
        "duration": duration,
        "fps": fps,
        "video_type": video_type,
        "num_anchors": len(anchors),
        "anchor_times": anchors,
        "frames_per_anchor": frames_per_anchor,
        "total_yolo_frames": total_yolo_frames,
        "window_stable_counts": window_stable_counts,
        "final_count": final_count
    }

    return final_count, debug_info

In [12]:
def fix_video_path(path):
    return path.replace(
        "/data/shared/private",
        "/redgiant/data/shared/private"
    )

In [13]:
def save_results_pretty_compact_lists(results, output_json):
    text = json.dumps(results, indent=2)

    for key in ["anchor_times", "window_stable_counts"]:
        pattern = re.compile(
            rf'("{key}": )\[\n\s+([^\]]*?)\n\s+\]',
            re.DOTALL
        )

        def compact_array(match):
            prefix = match.group(1)
            array_body = match.group(2)

            values = [
                x.strip().rstrip(",")
                for x in array_body.splitlines()
                if x.strip()
            ]

            return prefix + "[" + ", ".join(values) + "]"

        text = pattern.sub(compact_array, text)

    with open(output_json, "w") as f:
        f.write(text)

This is the script that runs from .2 to .3 threshold 

In [14]:
import json
import time
import pandas as pd
import numpy as np

metadata_path = "/supernova/data/home/aryah/raw_videos.json"
output_json = "/supernova/data/home/aryah/yolo26_under45_first100_conf025.json"


GROUND_TRUTH_COUNTS = {
    3766: 1,
    5505: 1,
    196669: 1,
    196810: 1,
    134: 5,
    151: 1,
    260: 2,
    811: 1,
    3776: 1,
    3871: 1,
    4596: 2,

    24: 2,
    28: 2,
    29: 1,
    77: 1,
    86: 2,
    123: 2,
    146: 2,
    152: 2,
    172: 1,
    176: 1,
    190: 1,
    195: 2,
    198: 1,
    245: 1,
    246: 1,
    269: 1,
    322: 1,
    330: 2,
    334: 2,
    422: 2,
    411: 1,

    253: 2,
    255: 2,
    261: 1,   
    262: 3,
    271: 4,
    275: 1,
    286: 3,
    321: 1,
    366: 8,
    417: 1,
    418: 1,
    419: 3,

    2151: 3,
    1716: 2,
    1673: 1,
    1552: 2,
    1458: 2,
    1445: 2,
    1399: 1,
    1356: 2,
    1328: 1,
    1255: 1,
    1230: 3,
    1229: 3,
    944: 3,
    723: 3,
    722: 3,
    485: 4,
    478: 4,
    472: 3,
    453: 3,
    442: 3,
    439: 3,
}

TEST_VIDEO_IDS = set(GROUND_TRUTH_COUNTS.keys())

COVERAGE = 0.40
WINDOW_SIZE = 2.0
FRAMES_PER_ANCHOR = 10
MAX_ANCHORS = None
DEVICE = 0

THRESHOLDS = [0.31, 0.315, 0.32, 0.325, 0.33, 0.335, 0.34]

with open(metadata_path, "r") as f:
    all_videos = json.load(f)

videos = [v for v in all_videos if int(v["video_id"]) in TEST_VIDEO_IDS]

print(f"Found {len(videos)} validation videos:")
print([v["video_id"] for v in videos])

all_threshold_summaries = []

for CONF_THRESHOLD in THRESHOLDS:
    print("\n==============================")
    print(f"Running validation for CONF_THRESHOLD = {CONF_THRESHOLD}")
    print("==============================")

    conf_name = str(CONF_THRESHOLD).replace(".", "p")
    output_json = f"/supernova/data/home/aryah/yolo26_validation_conf{conf_name}.json"
    results = []

    for i, video in enumerate(videos):
        video_id = int(video["video_id"])
        video_identifier = video.get("video_identifier", "")
        video_path = fix_video_path(video["video_path"])
        duration = float(video["duration"])
        manual_count = GROUND_TRUTH_COUNTS.get(video_id, None)

        print(f"[{i+1}/{len(videos)}] video_id={video_id}")

        start_time = time.time()

        try:
            count, info = estimate_num_dogs_yolo26(
                video_path=video_path,
                model=model,
                duration=duration,
                coverage=COVERAGE,
                window_size=WINDOW_SIZE,
                frames_per_anchor=FRAMES_PER_ANCHOR,
                conf_threshold=CONF_THRESHOLD,
                max_anchors=MAX_ANCHORS,
                device=DEVICE
            )

            error = info.get("error", "")

        except Exception as e:
            count = -1
            info = {}
            error = str(e)

        runtime_seconds = time.time() - start_time
        total_yolo_frames = info.get("total_yolo_frames", 0)

        result = {
            "video_id": video_id,
            "video_identifier": video_identifier,
            "video_path": video_path,
            "duration": duration,

            "manual_count": manual_count,
            "num_dogs_visible_estimated": count,
            "correct": count == manual_count if manual_count is not None and count >= 0 else False,
            "absolute_error": abs(count - manual_count) if manual_count is not None and count >= 0 else None,

            "runtime_seconds": runtime_seconds,
            "total_yolo_frames": total_yolo_frames,
            "effective_fps": total_yolo_frames / runtime_seconds if runtime_seconds > 0 else 0,

            "fps": info.get("fps", None),
            "video_type": info.get("video_type", None),
            "num_anchors": info.get("num_anchors", None),
            "anchor_times": info.get("anchor_times", []),
            "frames_per_anchor": info.get("frames_per_anchor", FRAMES_PER_ANCHOR),

            "coverage": COVERAGE,
            "window_size": WINDOW_SIZE,
            "conf_threshold": CONF_THRESHOLD,

            "window_stable_counts": info.get("window_stable_counts", []),

            "error": error
        }

        results.append(result)

        # Save after every video so progress is not lost
        save_results_pretty_compact_lists(results, output_json)

    df = pd.DataFrame(results)
    valid = df[df["error"].fillna("") == ""]

    accuracy = valid["correct"].mean()
    mean_abs_error = valid["absolute_error"].mean()
    correct_count = int(valid["correct"].sum())

    all_threshold_summaries.append({
        "conf_threshold": CONF_THRESHOLD,
        "correct_count": correct_count,
        "total_videos": len(valid),
        "accuracy": accuracy,
        "mean_absolute_error": mean_abs_error,
        "output_json": output_json
    })

summary_df = pd.DataFrame(all_threshold_summaries)

print("\nFinal threshold comparison:")
print(summary_df)

best = summary_df.sort_values(
    by=["accuracy", "mean_absolute_error"],
    ascending=[False, True]
).iloc[0]

print("\nBest threshold:")
print(best)

Found 65 validation videos:
[24, 28, 29, 77, 86, 123, 134, 146, 151, 152, 172, 176, 190, 195, 198, 245, 246, 253, 255, 260, 261, 262, 269, 271, 275, 286, 321, 322, 330, 334, 366, 411, 417, 418, 419, 422, 439, 442, 453, 472, 478, 485, 722, 723, 811, 944, 1229, 1230, 1255, 1328, 1356, 1399, 1445, 1458, 1552, 1673, 1716, 2151, 3766, 3776, 3871, 4596, 5505, 196669, 196810]

Running validation for CONF_THRESHOLD = 0.31
[1/65] video_id=24
[2/65] video_id=28
[3/65] video_id=29
[4/65] video_id=77
[5/65] video_id=86
[6/65] video_id=123
[7/65] video_id=134
[8/65] video_id=146
[9/65] video_id=151
[10/65] video_id=152
[11/65] video_id=172
[12/65] video_id=176
[13/65] video_id=190
[14/65] video_id=195
[15/65] video_id=198
[16/65] video_id=245
[17/65] video_id=246
[18/65] video_id=253
[19/65] video_id=255
[20/65] video_id=260
[21/65] video_id=261
[22/65] video_id=262
[23/65] video_id=269
[24/65] video_id=271
[25/65] video_id=275
[26/65] video_id=286
[27/65] video_id=321
[28/65] video_id=322
[29/65] 

[opus @ 0x5dac3dd98340] Error parsing Opus packet header.
[opus @ 0x5dac3dd98340] Error parsing Opus packet header.
[opus @ 0x5dac3dd98340] Error parsing Opus packet header.


[50/65] video_id=1328


[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.


[51/65] video_id=1356


...

[53/65] video_id=1445


[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.
[opus @ 0x5dac3c64d2c0] Error parsing Opus packet header.


[54/65] video_id=1458
[55/65] video_id=1552


[opus @ 0x5dac3d819b80] Error parsing Opus packet header.
[opus @ 0x5dac3d819b80] Error parsing Opus packet header.
[opus @ 0x5dac3d819b80] Error parsing Opus packet header.


[56/65] video_id=1673
[57/65] video_id=1716
[58/65] video_id=2151
[59/65] video_id=3766
[60/65] video_id=3776
[61/65] video_id=3871
[62/65] video_id=4596
[63/65] video_id=5505
[64/65] video_id=196669
[65/65] video_id=196810

Final threshold comparison:
   conf_threshold  correct_count  total_videos  accuracy  mean_absolute_error  \
0           0.310             31            65  0.476923             0.723077   
1           0.315             33            65  0.507692             0.692308   
2           0.320             34            65  0.523077             0.692308   
3           0.325             34            65  0.523077             0.676923   
4           0.330             34            65  0.523077             0.676923   
5           0.335             35            65  0.538462             0.707692   
6           0.340             35            65  0.538462             0.707692   

                                         output_json  
0  /supernova/data/home/aryah/yolo26_valida

This is the script that has a threshold of 2.5 and 1k vids

In [ ]:
import json
import time
import pandas as pd

metadata_path = "/supernova/data/home/aryah/raw_videos.json"
output_json = "/supernova/data/home/aryah/yolo26_under45_first100_conf025.json"

test_ids_path = "/supernova/data/home/aryah/first_1000_under100_video_ids.json"

with open(test_ids_path, "r") as f:
    TEST_VIDEO_IDS = set(json.load(f))
    
COVERAGE = 0.40
WINDOW_SIZE = 2.0
FRAMES_PER_ANCHOR = 10
MAX_ANCHORS = None
DEVICE = 0

CONF_THRESHOLD = 0.33

with open(metadata_path, "r") as f:
    all_videos = json.load(f)

videos = [v for v in all_videos if int(v["video_id"]) in TEST_VIDEO_IDS]

# Keep same order as raw_videos.json
videos = sorted(videos, key=lambda v: int(v["video_id"]))

print(f"Found {len(videos)} videos to run")
print(f"Using CONF_THRESHOLD = {CONF_THRESHOLD}")
print(f"Output: {output_json}")
print([int(v["video_id"]) for v in videos])

results = []

for i, video in enumerate(videos):
    video_id = int(video["video_id"])
    video_identifier = video.get("video_identifier", "")
    video_path = fix_video_path(video["video_path"])
    duration = float(video["duration"])

    print("\n==============================")
    print(f"[{i+1}/{len(videos)}] video_id={video_id}")
    print(f"duration={duration:.2f}")
    print("==============================")

    start_time = time.time()

    try:
        count, info = estimate_num_dogs_yolo26(
            video_path=video_path,
            model=model,
            duration=duration,
            coverage=COVERAGE,
            window_size=WINDOW_SIZE,
            frames_per_anchor=FRAMES_PER_ANCHOR,
            conf_threshold=CONF_THRESHOLD,
            max_anchors=MAX_ANCHORS,
            device=DEVICE
        )

        error = info.get("error", "")

    except Exception as e:
        count = -1
        info = {}
        error = str(e)

    runtime_seconds = time.time() - start_time
    total_yolo_frames = info.get("total_yolo_frames", 0)

    result = {
        "video_id": video_id,
        "video_identifier": video_identifier,
        "video_path": video_path,
        "duration": duration,

        "num_dogs_visible_estimated": count,

        "runtime_seconds": runtime_seconds,
        "total_yolo_frames": total_yolo_frames,
        "effective_fps": total_yolo_frames / runtime_seconds if runtime_seconds > 0 else 0,

        "fps": info.get("fps", None),
        "video_type": info.get("video_type", None),
        "num_anchors": info.get("num_anchors", None),
        "anchor_times": info.get("anchor_times", []),
        "frames_per_anchor": info.get("frames_per_anchor", FRAMES_PER_ANCHOR),

        "coverage": COVERAGE,
        "window_size": WINDOW_SIZE,
        "conf_threshold": CONF_THRESHOLD,

        "window_stable_counts": info.get("window_stable_counts", []),

        "error": error
    }

    results.append(result)

    # Save after every video so progress is not lost
    save_results_pretty_compact_lists(results, output_json)

    print(f"Estimated dogs: {count}")
    print(f"Runtime seconds: {runtime_seconds:.2f}")
    print(f"Total YOLO frames: {total_yolo_frames}")
    print(f"Error: {error}")

df = pd.DataFrame(results)

print("\nFinished running selected videos.")
print(f"Saved results to: {output_json}")

print("\nPreview:")
print(df[
    [
        "video_id",
        "duration",
        "num_dogs_visible_estimated",
        "video_type",
        "num_anchors",
        "total_yolo_frames",
        "runtime_seconds",
        "error"
    ]
])

print("\nDog count distribution:")
print(df["num_dogs_visible_estimated"].value_counts().sort_index())

print("\nErrors:")
print(df[df["error"].fillna("") != ""][
    [
        "video_id",
        "video_path",
        "error"
    ]
])

script to run the .2 to .3 and see whats better

In [ ]:
import json
import os
import pandas as pd

summaries = []

for t in range(30, 41):
    conf = t / 100
    path = f"/supernova/data/home/aryah/yolo26_validation_confOp{t:03d}.json"

    if not os.path.exists(path):
        print("Missing:", path)
        continue

    with open(path, "r") as f:
        results = json.load(f)

    df = pd.DataFrame(results)
    valid = df[df["error"].fillna("") == ""]

    summaries.append({
        "conf_threshold": conf,
        "correct": int(valid["correct"].sum()),
        "total": len(valid),
        "accuracy": valid["correct"].mean(),
        "mean_absolute_error": valid["absolute_error"].mean()
    })

summary_df = pd.DataFrame(summaries)

print(summary_df)

best = summary_df.sort_values(
    by=["accuracy", "mean_absolute_error"],
    ascending=[False, True]
).iloc[0]

print("\nBest threshold:")
print(best)

script to see which of the 100 vids are 2 dogs or more

In [43]:
import json
import pandas as pd

output_json = "/supernova/data/home/aryah/yolo26_under45_first100_conf025.json"

with open(output_json, "r") as f:
    results = json.load(f)

df = pd.DataFrame(results)

valid = df[df["error"].fillna("") == ""].copy()

more_than_2 = valid[valid["num_dogs_visible_estimated"] > 2].copy()

ids_more_than_2 = sorted(more_than_2["video_id"].astype(int).tolist())

print(ids_more_than_2)
print(f"\nTotal videos estimated > 2 dogs: {len(ids_more_than_2)}")

[253, 255, 261, 262, 271, 275, 286, 321, 366, 411, 417, 418, 419, 420, 439, 442, 453, 472, 478, 485, 722, 723, 731, 944, 1229, 1230, 1255, 1328, 1356, 1399, 1445, 1458, 1552, 1673, 1716, 2151]

Total videos estimated > 2 dogs: 36


In [25]:
import json
import pandas as pd

THRESHOLDS = [0.310, 0.315, 0.320, 0.325, 0.330, 0.335, 0.340]

summary_rows = []
all_detail_rows = []

for conf in THRESHOLDS:
    conf_name = str(conf).replace(".", "p")
    json_path = f"/supernova/data/home/aryah/yolo26_validation_conf{conf_name}.json"

    with open(json_path, "r") as f:
        results = json.load(f)

    detail_rows = []

    for r in results:
        video_id = int(r["video_id"])
        truth = int(GROUND_TRUTH_COUNTS[video_id])
        pred = int(r["num_dogs_visible_estimated"])
        error = pred - truth

        # Keep every row, but mark pred == 0 as excluded from off-by-one stats
        used_for_off_by_one_stats = pred != 0

        if pred == 0:
            status = "excluded_pred_0"
        elif error == 0:
            status = "correct"
        elif error == -1:
            status = "undercount_by_1"
        elif error == 1:
            status = "overcount_by_1"
        else:
            status = "other_wrong"

        detail_rows.append({
            "conf_threshold": conf,
            "video_id": video_id,
            "truth": truth,
            "predicted": pred,
            "error_pred_minus_truth": error,
            "used_for_off_by_one_stats": used_for_off_by_one_stats,
            "status": status,
        })

    df = pd.DataFrame(detail_rows)

    # Keep df unchanged for display, but use only this filtered version for metrics
    metric_df = df[df["used_for_off_by_one_stats"]].copy()

    correct = (metric_df["status"] == "correct").sum()
    under_by_1 = (metric_df["status"] == "undercount_by_1").sum()
    over_by_1 = (metric_df["status"] == "overcount_by_1").sum()

    correct_plus_minus_1 = correct + under_by_1 + over_by_1

    total_original = len(df)
    excluded_pred_0 = (df["status"] == "excluded_pred_0").sum()
    total_used_for_stats = len(metric_df)

    summary_rows.append({
        "conf_threshold": conf,
        "total_original_videos": total_original,
        "excluded_pred_0": excluded_pred_0,
        "total_used_for_stats": total_used_for_stats,

        "correct": correct,
        "undercount_by_1": under_by_1,
        "overcount_by_1": over_by_1,
        "correct_plus_minus_1": correct_plus_minus_1,

        "accuracy_exact_used_only": correct / total_used_for_stats if total_used_for_stats else 0,
        "accuracy_correct_plus_minus_1_used_only": correct_plus_minus_1 / total_used_for_stats if total_used_for_stats else 0,
    })

    all_detail_rows.append(df)

summary_df = pd.DataFrame(summary_rows)
details_df = pd.concat(all_detail_rows, ignore_index=True)

print("Summary: pred == 0 rows are shown but excluded from off-by-one stats")
display(summary_df)

print("All rows, including pred == 0:")
display(details_df)

Summary: pred == 0 rows are shown but excluded from off-by-one stats


,conf_threshold,total_original_videos,excluded_pred_0,total_used_for_stats,correct,undercount_by_1,overcount_by_1,correct_plus_minus_1,accuracy_exact_used_only,accuracy_correct_plus_minus_1_used_only
0,0.310,65,2,63,31,7,18,56,0.492063,0.888889
1,0.315,65,2,63,33,7,16,56,0.523810,0.888889
2,0.320,65,2,63,34,6,15,55,0.539683,0.873016
3,0.325,65,2,63,34,8,14,56,0.539683,0.888889
4,0.330,65,2,63,34,9,13,56,0.539683,0.888889
5,0.335,65,3,62,35,6,12,53,0.564516,0.854839
6,0.340,65,3,62,35,6,12,53,0.564516,0.854839


All rows, including pred == 0:


,conf_threshold,video_id,truth,predicted,error_pred_minus_truth,used_for_off_by_one_stats,status
0,0.31,24,2,2,0,True,correct
1,0.31,28,2,1,-1,True,undercount_by_1
2,0.31,29,1,1,0,True,correct
3,0.31,77,1,1,0,True,correct
4,0.31,86,2,2,0,True,correct
...,...,...,...,...,...,...,...
450,0.34,3871,1,1,0,True,correct
451,0.34,4596,2,2,0,True,correct
452,0.34,5505,1,1,0,True,correct
453,0.34,196669,1,1,0,True,correct
